# 24 — G3TI: ablasi diskriminator tekstur dan dua seed tambahan, p100

Tujuan (keputusan pengguna, 2 Okt 2026). Kernel 23 menjadikan G3TI kandidat MRT-GAN: kompositing hanya saat ekspor
(`composite_mask=0` saat latih) dan identity 0,5. Notebook ini menguji dua hal:

- `G3TI_noT`: G3TI dengan `lambda_texture=0`, menjadi shared baseline baru. Pertanyaannya: apakah diskriminator
  tekstur RGB masih membantu di atas basis baru?
- `G3TIs12`, `G3TIs13`: G3TI dengan seed 12 dan 13, untuk mengukur stabilitas antar-seed.

Selain itu identik dengan G3TI seed 11 (kernel 23). Jadwal 20 epoch, pool p100 dari kernel 13. Tidak ada detektor.
Test tidak dibaca.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/g3ti_ablation_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
# A previous stage's kernel output also carries a code copy next to study/; only the code dataset counts.
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
# Study (p100 crops and configs) from kernel 13 output; three G3TI ablation configs written next to G0-G3.
if torch.cuda.device_count()<1: raise RuntimeError('GPU required')
print('GPUs', torch.cuda.device_count())
source=unique('study.json')
STUDY=WORK/'study'
shutil.copytree(source,STUDY)
for p in [STUDY,*STUDY.rglob('*')]: p.chmod(p.stat().st_mode|0o200)
P100=STUDY/'p100'
sys.path.insert(0,str(HERE))
from melon_cyclegan import texture_study as s
from melon_cyclegan import texture_study_eval as e
scope,m=s.verify_scope(STUDY,100,DATASET)
g0=json.loads((P100/'configs/G0.json').read_text()); g3=json.loads((P100/'configs/G3.json').read_text())
assert g0['subset_digest']==g3['subset_digest']==m['subset_digest']
g3ti=dict(g3,composite_mask=0,lambda_identity=0.5)
EXTRA={'G3TI_noT':dict(g3ti,lambda_texture=0.0,experiment='p100_G3TI_noT_seed11',rung='G3TI_noT',
        purpose='G3TI without texture discriminator (new shared baseline); train only',
        note='G3TI seed 11 with lambda_texture 0; export pasted through the fruit mask')}
for seed in (12,13):
    EXTRA[f'G3TIs{seed}']=dict(g3ti,experiment=f'p100_G3TI_seed{seed}',rung=f'G3TIs{seed}',seed=seed,
        controlled_init_seed=seed,texture_seed=seed*1000+1,purpose='G3TI seed repeat; train only',
        note='G3TI except seed; export pasted through the fruit mask')
for arm,c in EXTRA.items(): assert c['model']=='melon_texture' and c['composite_mask']==0 and c['lambda_identity']==0.5
for arm,cfg_ in EXTRA.items():
    target=P100/'configs'/f'{arm}.json'
    if target.exists(): raise ValueError(f'{target} exists')
    target.write_text(json.dumps(cfg_,indent=2)+'\n')
    print(arm,{k:cfg_[k] for k in ('model','seed','composite_mask','lambda_color_margin','diffaug','lambda_identity','ema_decay')})


In [ ]:
# Train: GPU0 runs G3TI_noT then G3TIs12, GPU1 runs G3TIs13 (sequential on one GPU if only one is present).
import shlex, time
def train_cmd(arm):
    return [sys.executable,str(HERE/'tools/train_cyclegan.py'),'--config',str(P100/'configs'/f'{arm}.json'),
            '--dataroot',str(P100/'crops'),'--checkpoints-dir',str(P100/'checkpoints'),'--','--time_budget_minutes','30']
(P100/'logs').mkdir(exist_ok=True)
def chain(arms,gpu):
    env=(f"CUDA_VISIBLE_DEVICES={gpu} PYTHONPATH={shlex.quote(str(HERE))} MELON_DATASET={shlex.quote(str(DATASET))} "
         f"MELON_ASSETS={shlex.quote(str(ASSETS))} OMP_NUM_THREADS=2")
    cmd=' && '.join(env+' '+shlex.join(train_cmd(a))+' > '+shlex.quote(str(P100/'logs'/f'{a}.log'))+' 2>&1' for a in arms)
    return subprocess.Popen(cmd,shell=True,cwd=str(HERE))
started=time.time()
plan=[(['G3TI_noT','G3TIs12'],0),(['G3TIs13'],1)] if torch.cuda.device_count()>=2 else [(['G3TI_noT','G3TIs12','G3TIs13'],0)]
codes=[p.wait() for p in [chain(a,g) for a,g in plan]]
print('return codes',codes,'wall minutes',round((time.time()-started)/60,1))
for arm in EXTRA: print('---',arm); print((P100/'logs'/f'{arm}.log').read_text()[-1500:])
if any(codes): raise SystemExit('training failed')
for arm,c in EXTRA.items():
    loss=json.loads((P100/'checkpoints'/c['experiment']/'loss.json').read_text())
    if loss['stopped'] is not None or loss['epochs']!=c['n_epochs']+c['n_epochs_decay']: raise SystemExit(f'{arm} partial run')

# Pre-registered H-T check: mean losses over epochs 11-20 (LSGAN equilibrium 0.25).
import re
LOSS_MEANS={}
for arm in EXTRA:
    rows=[dict((k,float(v)) for k,v in re.findall(r'(\w+):([-\d.]+)',l))
          for l in (P100/'logs'/f'{arm}.log').read_text().splitlines()
          if (mm:=re.match(r'epoch (\d+) iter',l)) and int(mm.group(1))>=11]
    LOSS_MEANS[arm]={k:round(sum(r[k] for r in rows)/len(rows),3) for k in ('D_A','D_B','G_A','G_B','idt_A','idt_B','texture_D_A','texture_D_B') if k in rows[0]}
    LOSS_MEANS[arm]['n_rows']=len(rows)
    print(arm,LOSS_MEANS[arm])


In [ ]:
# Export candidates (EMA, pasted through the fruit mask as for G0-G3) and train-only diagnostics.
s.export_candidates(STUDY,100,list(EXTRA),DATASET,'cuda')
e.diagnostics(STUDY,100,list(EXTRA),'cuda')
ref=json.loads((P100/'diagnostics/G3.json').read_text())['synthetic'] if (P100/'diagnostics/G3.json').exists() else None
for arm in EXTRA:
    d=json.loads((P100/'diagnostics'/f'{arm}.json').read_text())['synthetic']
    print(arm,{k:round(d[k],4) for k in ('nyquist_luma','chroma_energy','luma_energy','lbp_chi2')})
if ref: print('G3 seed11',{k:round(ref[k],4) for k in ('nyquist_luma','chroma_energy','luma_energy','lbp_chi2')})


In [ ]:
# Keep only what local FID/KID, review and the paper need; drop crops, full frames and other arms.
OUT=WORK/'result'; OUT.mkdir()
for arm,c in EXTRA.items():
    ck=P100/'checkpoints'/c['experiment']; dst=OUT/'checkpoints'/c['experiment']; dst.mkdir(parents=True)
    for name in ('loss.json','melon_identity.json','latest_net_G_B_ema.pth','latest_net_G_A_ema.pth'):
        if (ck/name).exists(): shutil.copy2(ck/name,dst/name)
    cand=P100/'candidates'/arm; cdst=OUT/'candidates'/arm; cdst.mkdir(parents=True)
    for sub in ('images','masks'): shutil.copytree(cand/sub,cdst/sub)
    for name in ('records.json','binding.json'): shutil.copy2(cand/name,cdst/name)
    shutil.copy2(P100/'configs'/f'{arm}.json',OUT/f'config_{arm}.json')
    shutil.copy2(P100/'diagnostics'/f'{arm}.json',OUT/f'diagnostics_{arm}.json')
    shutil.copy2(P100/'logs'/f'{arm}.log',OUT/f'train_{arm}.log')
(OUT/'summary.json').write_text(json.dumps(dict(arms=list(EXTRA),loss_means_ep11_20=LOSS_MEANS,subset_digest=m['subset_digest'],
    test_used=False,wall_minutes=round((time.time()-started)/60,1)),indent=1))
shutil.rmtree(STUDY,ignore_errors=True); shutil.rmtree(WORK/'code',ignore_errors=True)
total=sum(p.stat().st_size for p in WORK.rglob('*') if p.is_file())
print('output files',sum(1 for p in WORK.rglob('*') if p.is_file()),'MB',round(total/2**20))
